[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: pipeline de integración continua de una API

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

Las comprobaciones locales instalan las dependencias de la *API* en un entorno virtual temporal y por ello necesitan acceso a *PyPI*. No modifican el repositorio: trabajan sobre una copia.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Comprobar los archivos del pipeline

Asegurarse de que el repositorio de trabajo contiene el directorio `api/`, el `Dockerfile` en la raíz y el archivo `.github/workflows/ci.yml`. Si el *workflow* no aparece, revisar que el archivo esté en `.github/workflows/` y que la extensión sea `.yml` o `.yaml`.

La siguiente celda comprobará la existencia de los tres elementos y que todos los archivos de `.github/workflows` tengan una extensión que *GitHub* reconoce.

In [ ]:
for ruta in ("api", "Dockerfile", ".github/workflows/ci.yml"):
    existe = (RAIZ / ruta).exists()
    print(f"{ruta:28} {'existe' if existe else 'FALTA'}")
    assert existe

extensiones = {p.suffix for p in (RAIZ / ".github" / "workflows").iterdir()}
print("Extensiones en .github/workflows:", extensiones)
assert extensiones <= {".yml", ".yaml"}

## Ejercicio 2. Disparar el workflow CI con un push o un pull request

Hacer un *push* o abrir un *pull request* para disparar el *workflow* `CI`.

La siguiente celda mostrará los disparadores de `ci.yml`: cualquier *push*, los *pull requests* y la ejecución manual. Un *push* a cualquier rama, o un *pull request* sobre cualquier rama, basta para que el *job* `verificar` se ejecute.

In [ ]:
wf = u.cargar_workflow(RAIZ / ".github" / "workflows" / "ci.yml")
print(sorted(wf["on"]))
assert set(wf["on"]) == {"push", "pull_request", "workflow_dispatch"}

La siguiente celda creará una rama de trabajo, hará un *commit* vacío y la enviará. Después abrirá un *pull request* hacia `main` con el *CLI*. Se usa un *commit* vacío para no modificar ningún archivo.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
git switch --create prueba-ci
git commit --allow-empty --message "Dispara el workflow CI"
git push --set-upstream origin prueba-ci
gh pr create --base main --head prueba-ci \
  --title "Prueba del workflow CI" --body "Ejecución de comprobación del pipeline."

**Salida esperada.** `gh pr create` imprime la dirección del *pull request*. En la pestaña `Actions` aparece una ejecución de `CI` asociada a la rama `prueba-ci`.

## Ejercicio 3. Confirmar el job verificar y el artefacto

Confirmar que el *job* `verificar` termina en verde y que el artefacto `cobertura-pytest` aparece en el resumen de la ejecución.

El *job* `verificar` ejecuta, en este orden: instalar dependencias, *linter* (`ruff check` y `ruff format --check`), pruebas (`pytest`), publicación del artefacto y construcción de la imagen. Los tres primeros se pueden repetir en local, sobre una copia del repositorio.

La siguiente celda copiará el repositorio a una carpeta temporal y creará el entorno virtual con `requirements-dev.txt`, como el paso `Instalar dependencias` del *job*.

In [ ]:
COPIA = u.copiar_repositorio(RAIZ, u.directorio_temporal())
PYTHON = u.crear_entorno_api(COPIA / "api", COPIA.parent)
API = COPIA / "api"
print(u.ejecutar([str(PYTHON), "-m", "ruff", "--version"], API).stdout.strip())

La siguiente celda ejecutará los pasos `Linter (ruff)` y `Pruebas (pytest)` en la copia y comprobará que ambos terminan con código 0 y que `pytest` deja el archivo `coverage.xml` que después se publica como artefacto.

In [ ]:
for comando in (
    [str(PYTHON), "-m", "ruff", "check", "."],
    [str(PYTHON), "-m", "ruff", "format", "--check", "."],
):
    resultado = u.ejecutar(comando, API, comprobar=False)
    print(" ".join(comando[2:]), "->", resultado.returncode, resultado.stdout.strip().splitlines()[-1])
    assert resultado.returncode == 0

pruebas = u.ejecutar([str(PYTHON), "-m", "pytest"], API, comprobar=False)
print(pruebas.stdout.strip().splitlines()[-1])
assert pruebas.returncode == 0
assert (API / "coverage.xml").exists()

La siguiente celda consultará en *GitHub* el estado de la última ejecución de `CI`: la conclusión de cada *job* y los artefactos publicados.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
ID="$(gh run list --workflow CI --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run view "${ID}" --json status,conclusion,jobs --jq '{status, conclusion, jobs: [.jobs[] | {name, conclusion}]}'
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
gh api "repos/${REPO}/actions/runs/${ID}/artifacts" --jq '[.artifacts[].name]'

**Salida esperada.** `status: completed`, `conclusion: success`, el *job* `verificar` con `success` y la lista `["cobertura-pytest"]`.

## Ejercicio 4. Provocar un fallo a propósito

Alterar una prueba de forma temporal, volver a ejecutar y comprobar que el *workflow* falla. El informe de cobertura, si llegó a generarse, puede seguir apareciendo como artefacto gracias a `if: always()`.

*El enunciado original no indica qué prueba alterar. Esta solución cambia el resultado esperado de `test_health_ok` de `"ok"` a `"error"`.*

La siguiente celda cambiará, solo en la copia temporal, la aserción de `tests/test_health.py` y volverá a ejecutar *pytest*. Se espera un código de salida distinto de cero, una prueba fallida y, aun así, un `coverage.xml` generado.

In [ ]:
prueba = API / "tests" / "test_health.py"
original = prueba.read_text(encoding="utf-8")
alterado = original.replace('assert cuerpo["status"] == "ok"', 'assert cuerpo["status"] == "error"', 1)
assert alterado != original
prueba.write_text(alterado, encoding="utf-8")
(API / "coverage.xml").unlink()

fallo = u.ejecutar([str(PYTHON), "-m", "pytest"], API, comprobar=False)
print("Código de salida:", fallo.returncode)
print([linea for linea in fallo.stdout.splitlines() if linea.startswith("FAILED")])
print("¿Existe coverage.xml?", (API / "coverage.xml").exists())
assert fallo.returncode != 0
assert (API / "coverage.xml").exists()

Por eso el paso `Publicar informe de cobertura` lleva `if: always()`: sin esa condición, el artefacto no se publicaría cuando las pruebas fallan, que es justo cuando más sirve. La copia se descarta al terminar; el repositorio original no cambió.

La siguiente celda restaurará la prueba en la copia y comprobará que el resultado vuelve a ser correcto.

In [ ]:
prueba.write_text(original, encoding="utf-8")
recuperado = u.ejecutar([str(PYTHON), "-m", "pytest"], API, comprobar=False)
assert recuperado.returncode == 0
print(recuperado.stdout.strip().splitlines()[-1])

La siguiente celda repetirá la práctica en *GitHub*: cambiará la aserción en la rama `prueba-ci`, enviará el cambio y mostrará que la ejecución falla pero conserva el artefacto. Al final, descartará el cambio y borrará la rama de prueba.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
git switch prueba-ci
sed -i 's/assert cuerpo\["status"\] == "ok"/assert cuerpo["status"] == "error"/' api/tests/test_health.py
git commit --all --message "Rompe una prueba a propósito"
git push
sleep 10
ID="$(gh run list --branch prueba-ci --workflow CI --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run watch "${ID}" || true
gh run view "${ID}" --json conclusion --jq .conclusion
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
gh api "repos/${REPO}/actions/runs/${ID}/artifacts" --jq '[.artifacts[].name]'
git switch main
git push origin --delete prueba-ci
git branch --delete --force prueba-ci

**Salida esperada.** La conclusión es `failure`, el *job* `verificar` falla en el paso `Pruebas (pytest)` y la lista de artefactos sigue incluyendo `cobertura-pytest`. El *pull request* abierto en el ejercicio 2 se cierra solo al borrar la rama.

## Ejercicio 5. Localizar el job publicar-imagen

En un *push* a `main`, localizar además el *job* `publicar-imagen` (se estudia en el cuaderno 15).

La siguiente celda leerá del archivo las tres condiciones que hacen que `publicar-imagen` solo corra tras un `verificar` verde, solo en un *push* a `main` y con permiso de escritura únicamente en paquetes.

In [ ]:
job = wf["jobs"]["publicar-imagen"]
print("needs      :", job["needs"])
print("if         :", job["if"])
print("permissions:", job["permissions"])
assert job["needs"] == "verificar"
assert "github.event_name == 'push'" in job["if"] and "refs/heads/main" in job["if"]
assert job["permissions"] == {"contents": "read", "packages": "write"}
assert "permissions" not in wf["jobs"]["verificar"]  # hereda contents: read

La siguiente celda mostrará los *jobs* de la última ejecución de `CI` en `main`. En un *push* a `main` deben aparecer `verificar` y `publicar-imagen`; en un *pull request*, solo `verificar`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
gh run list --workflow CI --branch main --event push --limit 1 --json databaseId --jq '.[0].databaseId' \
  | xargs -I{} gh run view {} --json jobs --jq '.jobs[] | {name, conclusion}'

**Salida esperada.** Dos objetos: `verificar` con `success` y `publicar-imagen` con `success`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>